# 6. Q-Learning with SeqTO-v1

## 6.1 Why Q-Learning for SeqTO-v1?

Chapter 5 attacked SeqTO-v1 with a Genetic Algorithm — a population-based, gradient-free metaheuristic that builds each iron mask from scratch every generation. This chapter takes the complementary route: an **experience-based** optimiser that builds up a value estimate over many episodes and exploits it to pick good actions one step at a time. The two methods are tested on the same MDP, with the same FEMM-based reward, so the comparison isolates the search strategy.

Q-learning fits SeqTO-v1 for four reasons:

1. **Model-free.** It needs no analytical model of how the magnetic field responds to iron deposits — just a reward signal from `seqTO_env.calculate_reward()`.
2. **Discrete action space.** Four actions (`RIGHT`, `LEFT`, `UP`, `DOWN`) per step exactly match the tabular Q-learning regime — no neural function approximator required.
3. **Sequential decision matches the SeqTO chromosome.** Every Q-learning episode *is* an action sequence; the connectivity guarantee that the GA chromosome enjoys (Section 5.1) carries over for free.
4. **Cheap incremental updates.** Each FEMM call updates a single $Q(s, a)$ entry; total memory grows only with states actually visited.

The implementation in this directory is [`q_learn_improved.py`](q_learn_improved.py). It uses a **compact 6-feature state representation** that keeps the Q-table tractable (3⁶ = 729 max states) while preserving enough Markov structure for convergent learning.

## 6.2 Q-Learning Foundations

This section gives a self-contained primer. Readers familiar with tabular Q-learning can skip to Section 6.3.

### 6.2.1 The MDP Setup

A Markov Decision Process is the tuple $\langle \mathcal{S}, \mathcal{A}, P, R, \gamma \rangle$:

- $\mathcal{S}$ — set of states (here, encodings of the controller's situation in the design window)
- $\mathcal{A} = \{0, 1, 2, 3\}$ — actions (RIGHT/LEFT/UP/DOWN)
- $P(s' \mid s, a)$ — transition kernel (deterministic in SeqTO-v1: the controller moves and the brush deposits iron)
- $R(s, a, s')$ — reward (here, the FEMM-evaluated mean $\lvert B \rvert$ at six armature sample points)
- $\gamma \in [0, 1)$ — discount factor (here $\gamma = 0.9$)

### 6.2.2 The Action-Value Function and Bellman Update

The **action-value function** $Q^\pi(s, a)$ is the expected discounted return from taking action $a$ in state $s$ and then following policy $\pi$. Q-learning estimates the *optimal* $Q^\star$ directly via the Bellman optimality equation:

$$Q^\star(s, a) = \mathbb{E}\bigl[ R(s, a, s') + \gamma \max_{a'} Q^\star(s', a') \bigr]$$

Sample-based, off-policy temporal-difference (TD) updates make this tractable. After every transition $(s, a, r, s')$:

$$Q(s, a) \;\leftarrow\; Q(s, a) + \alpha \bigl[\, r + \gamma \max_{a'} Q(s', a') - Q(s, a) \,\bigr]$$

where $\alpha \in (0, 1]$ is the learning rate. The bracketed quantity is the **TD error**. The agent moves $Q(s, a)$ a fraction $\alpha$ toward the bootstrapped target — toward observed reward plus the best estimate of future return.

### 6.2.3 ε-Greedy Exploration

If the agent always picked $\operatorname{argmax}_a Q(s, a)$ it would get stuck on the first non-trivial trajectory. **ε-greedy** mixes exploration and exploitation: with probability $\varepsilon$ pick a random action, otherwise pick the greedy one. $\varepsilon$ starts high (uniform random) and decays toward a small floor so that the late-training agent mostly exploits its learned values. This implementation uses **multiplicative decay** $\varepsilon_{t+1} = 0.97 \cdot \varepsilon_t$, clamped to $\varepsilon_{\min} = 0.05$, which reaches the floor near episode 115 of 200.

### 6.2.4 Tabular Storage

When $\mathcal{S}$ is small, $Q$ is stored as a lookup table — concretely, a sparse Python `defaultdict` keyed by the state tuple, with a length-4 numpy array per state. Updates are $O(1)$. Memory grows only with the set of states actually visited.

## 6.3 SeqTO-v1 Q-Learning Formulation

| MDP element | SeqTO-v1 / C-core specifics |
|---|---|
| State $s$ | 6-feature compact tuple (see Section 6.4) |
| Action $a$ | $\{0, 1, 2, 3\}$ = $\{\text{RIGHT}, \text{LEFT}, \text{UP}, \text{DOWN}\}$ |
| Transition $P(s' \mid s, a)$ | Deterministic: controller moves, $3 \times 3$ brush deposits iron at the new position (subject to design-window and coil-exclusion masks) |
| Reward $R$ | `seqTO_env.calculate_reward(iron_mask)` — mean $\lvert B \rvert$ at 6 armature sample points × 1000 |
| Terminal | `step_count >= max_steps` (34) or iron count exceeds $K_2 = 180$ |
| Discount $\gamma$ | $0.9$ |
| Learning rate $\alpha$ | $0.1$ |

The reward is the same scalar consumed by the GA in Chapter 5, so the comparison between gradient-free GA and experience-based Q-learning is on identical objective landscapes.

**Cumulative episode reward.** Each step produces a non-negative reward — the agent's episode-return is the *sum* of per-step mean-$\lvert B \rvert$ values rather than a single final fitness. This matters for interpreting magnitudes later: a 200-episode run sees episode rewards in the thousands, not the tens.

## 6.4 Compact 6-Feature State Representation

The most consequential design choice in this chapter is the state encoding. Two extremes are obvious and both fail:

- **Full iron mask** ($2^{630}$ possible states). Theoretically correct but every state is visited at most once — the Q-table never converges within any realistic episode budget.
- **Position-only** ($15 \times 21 = 315$ states). Tractable but Markov-violated: at the same $(r, c)$ the agent may be in radically different situations depending on which iron has already been deposited. The greedy policy ends up worse than random as $\varepsilon \to 0$.

`q_learn_improved.py` uses a hand-crafted **6-feature** state, each feature binned into $N_{\text{bins}} = 3$ levels (max $3^6 = 729$ states). Features:

| # | Feature | What it captures |
|---|---|---|
| 1 | `pos_r_bin`     | Controller row in the design window (low / mid / high) |
| 2 | `pos_c_bin`     | Controller column (left / mid / right) |
| 3 | `iron_frac_bin` | Iron budget consumed so far, relative to $K_2$ |
| 4 | `arm_iron_bin`  | Fraction of deposited iron in armature-facing columns (cols 19–25) |
| 5 | `vert_dist_bin` | Fraction of iron in the top half of the design window |
| 6 | `step_bin`      | Episode progress (early / mid / late) |

Features 4 and 5 encode *where in the window* the iron has been placed, not just *how much* — which is exactly the information missing from a position-only state. With $N_{\text{bins}} = 3$ the state-space is small enough that most states are visited many times across 200 episodes, so Q-values get repeated updates and the table actually converges.

The cell below defines the encoder, runs it on a hand-built sweep trajectory, and plots the resulting iron pattern alongside the 6-feature vector it produces:

In [ ]:
# 6-feature state encoder (verbatim logic from q_learn_improved.py)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

# Geometry (must match seqTO_env.py / q_learn_improved.py)
ROWS, COLS = 18, 35
COIL_LEN   = 9
COIL1_COLS = (2, 5)
COIL2_COLS = (11, 14)
ARM_ROWS, ARM_COLS = (0, 15), (27, 33)
DW_ROWS, DW_COLS   = (0, 15), (5, 26)
K1, K2     = 80, 180
MAX_STEPS  = 34
N_BINS     = 3

R_RANGE = DW_ROWS[1] - DW_ROWS[0]                  # 15
C_RANGE = DW_COLS[1] - DW_COLS[0]                  # 21
ARM_ZONE_START = DW_COLS[0] + (2 * C_RANGE // 3)   # col 19 — armature-facing zone
TOP_HALF_END   = DW_ROWS[0] + R_RANGE // 2         # row 10 — top/bottom split


def _bin(val, lo, hi, n):
    """Map val in [lo, hi] to integer bin index 0..n-1."""
    if hi <= lo:
        return 0
    return min(int((val - lo) * n / (hi - lo)), n - 1)


def get_state(iron_mask, pos_r, pos_c, step_count):
    """Compute compact 6-feature state tuple. Mirrors q_learn_improved.get_state()."""
    iron_count = int(np.sum(iron_mask))
    r_bin      = _bin(pos_r, DW_ROWS[0], DW_ROWS[1] - 1, N_BINS)
    c_bin      = _bin(pos_c, DW_COLS[0], DW_COLS[1] - 1, N_BINS)
    iron_b     = _bin(iron_count, 0, K2, N_BINS)
    arm_iron   = int(np.sum(iron_mask[DW_ROWS[0]:DW_ROWS[1], ARM_ZONE_START:DW_COLS[1]]))
    arm_frac   = arm_iron / max(iron_count, 1)
    arm_b      = _bin(arm_frac, 0.0, 1.0, N_BINS)
    top_iron   = int(np.sum(iron_mask[DW_ROWS[0]:TOP_HALF_END, DW_COLS[0]:DW_COLS[1]]))
    vert_frac  = top_iron / max(iron_count, 1)
    vert_b     = _bin(vert_frac, 0.0, 1.0, N_BINS)
    step_b     = _bin(step_count, 0, MAX_STEPS, N_BINS)
    return (r_bin, c_bin, iron_b, arm_b, vert_b, step_b)


# Pure-Python C-core topology decoder (3x3 brush) — same logic as ga_cCore.build_topology
ACTIONS = {0: ('RIGHT', 0, +1), 1: ('LEFT', 0, -1),
           2: ('UP',   -1,  0), 3: ('DOWN', +1,  0)}

def is_design_cell(i, j):
    in_dw    = DW_ROWS[0] <= i < DW_ROWS[1] and DW_COLS[0] <= j < DW_COLS[1]
    in_coil2 = i < COIL_LEN and COIL2_COLS[0] <= j < COIL2_COLS[1]
    return in_dw and not in_coil2

def build_topology(action_seq):
    mask = np.zeros((ROWS, COLS), dtype=int)
    pos_r, pos_c = DW_ROWS[0] + 1, DW_COLS[0] + 1
    for action in action_seq:
        _, dr, dc = ACTIONS[int(action)]
        new_r, new_c = pos_r + dr, pos_c + dc
        in_dw    = DW_ROWS[0] <= new_r < DW_ROWS[1] and DW_COLS[0] <= new_c < DW_COLS[1]
        in_coil2 = new_r < COIL_LEN and COIL2_COLS[0] <= new_c < COIL2_COLS[1]
        if in_dw and not in_coil2:
            pos_r, pos_c = new_r, new_c
            for di in range(-1, 2):
                for dj in range(-1, 2):
                    ri, ci = pos_r + di, pos_c + dj
                    if is_design_cell(ri, ci):
                        mask[ri, ci] = 1
    return mask, pos_r, pos_c


# Demo: build a half-sweep trajectory, encode its state, plot
demo_actions = [0]*7 + [3]*4 + [1]*5 + [3]*3        # RIGHT*7, DOWN*4, LEFT*5, DOWN*3 — 19 steps
mask, end_r, end_c = build_topology(demo_actions)
state              = get_state(mask, end_r, end_c, len(demo_actions))

# Region map for plotting
region = np.zeros((ROWS, COLS), dtype=int)
for i in range(ROWS):
    for j in range(COLS):
        if i < COIL_LEN and COIL1_COLS[0] <= j < COIL1_COLS[1]:    region[i, j] = 1
        elif i < COIL_LEN and COIL2_COLS[0] <= j < COIL2_COLS[1]:  region[i, j] = 2
        elif ARM_ROWS[0] <= i < ARM_ROWS[1] and ARM_COLS[0] <= j < ARM_COLS[1]: region[i, j] = 3
        elif is_design_cell(i, j):                                  region[i, j] = 4 if mask[i, j] else 5
        elif DW_ROWS[0] <= i < DW_ROWS[1] and DW_COLS[0] <= j < DW_COLS[1]: region[i, j] = 6

CLRS = ['#D5D8DC', '#2980B9', '#E67E22', '#27AE60', '#2C3E50', '#FEF9E7', '#FAD7A0']
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.imshow(region, cmap=ListedColormap(CLRS), vmin=0, vmax=6, origin='upper', aspect='equal')
ax.plot(end_c, end_r, 'rx', markersize=14, markeredgewidth=2.5, label='controller position')
ax.set_title(f'Demo trajectory after {len(demo_actions)} steps — '
             f'{int(mask.sum())} iron cells, ends at (row={end_r}, col={end_c})')
ax.set_xlabel('column'); ax.set_ylabel('row')
ax.legend(handles=[
    mpatches.Patch(fc='#2C3E50', label='Iron'),
    mpatches.Patch(fc='#FEF9E7', ec='grey', lw=0.5, label='Air (design window)'),
    mpatches.Patch(fc='#2980B9', label='Left coil'),
    mpatches.Patch(fc='#E67E22', label='Right coil'),
    mpatches.Patch(fc='#27AE60', label='Armature'),
], fontsize=8, loc='upper right')
plt.tight_layout(); plt.show()

print(f'6-feature state tuple: {state}')
print(f'  pos_r_bin     = {state[0]}  (controller row)')
print(f'  pos_c_bin     = {state[1]}  (controller col)')
print(f'  iron_frac_bin = {state[2]}  (iron count relative to K2={K2})')
print(f'  arm_iron_bin  = {state[3]}  (fraction of iron in armature-facing cols)')
print(f'  vert_dist_bin = {state[4]}  (fraction of iron in top half)')
print(f'  step_bin      = {state[5]}  (episode progress)')

## 6.5 Trainer and Episode Loop

The training loop is small enough to read in one go (full source in `q_learn_improved.py`, lines 132–195):

```python
Q = defaultdict(lambda: np.zeros(4))

def choose_action(state, epsilon):
    if np.random.random() < epsilon:
        return np.random.randint(4)
    return int(np.argmax(Q[state]))

def q_update(state, action, reward, next_state, done):
    target = reward if done else reward + GAMMA * np.max(Q[next_state])
    Q[state][action] += ALPHA * (target - Q[state][action])

def train():
    env     = CcoreSeqTOEnv(max_steps=MAX_STEPS)
    epsilon = EPSILON_START
    for ep in range(N_EPISODES):
        env.reset(random_start=RANDOM_START)
        state = get_state(env)
        ep_reward = 0.0
        for step in range(MAX_STEPS):
            action = choose_action(state, epsilon)
            _, reward, done, _ = env.step(action)
            next_state = get_state(env)
            q_update(state, action, reward, next_state, done)
            ep_reward += reward
            state = next_state
            if done: break
        epsilon = max(EPSILON_MIN, epsilon * EPSILON_DECAY)
```

Key choices:

- **`random_start=True`** (each episode resets the controller to a random cell inside the design window). Without it the agent over-trains on trajectories beginning at the default top-left and produces narrow, biased policies.
- **Episode reward is cumulative** — every step's mean-$\lvert B \rvert$ adds to `ep_reward`. The 200-episode reward trace below is therefore in the high thousands.
- **`q_update`** uses the standard TD target with `done=True` short-circuiting the bootstrap.

The cell below demonstrates the Q-update mechanics on a hand-crafted transition stream (no FEMM needed):

In [ ]:
# Q-update mechanics demo — five synthetic transitions, no FEMM
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np

Q = defaultdict(lambda: np.zeros(4))
ALPHA, GAMMA = 0.1, 0.9

# (state, action, reward, next_state, done)
transitions = [
    ((0, 0, 0, 0, 0, 0), 2, 50.0,  (0, 1, 0, 0, 0, 1), False),
    ((0, 1, 0, 0, 0, 1), 2, 120.0, (0, 2, 1, 0, 0, 2), False),
    ((0, 2, 1, 0, 0, 2), 0, 220.0, (1, 2, 1, 0, 0, 3), False),
    ((0, 0, 0, 0, 0, 0), 2, 50.0,  (0, 1, 0, 0, 0, 1), False),   # revisit
    ((0, 0, 0, 0, 0, 0), 2, 50.0,  (0, 1, 0, 0, 0, 1), False),   # revisit
]

print(f"{'t':>2}  {'state':<24}  {'a':<5}  {'reward':>8}  {'target':>8}  {'Q[s][a] (before -> after)':<28}")
for t, (s, a, r, ns, done) in enumerate(transitions):
    target = r if done else r + GAMMA * np.max(Q[ns])
    old    = Q[s][a]
    Q[s][a] += ALPHA * (target - Q[s][a])
    print(f'{t:>2}  {str(s):<24}  a={a:<3}  {r:>8.2f}  {target:>8.2f}  {old:7.4f}  ->  {Q[s][a]:.4f}')

print(f'\nQ-table size after 5 updates: {len(Q)} states')

# Visualise Q-values for the repeatedly-updated starting state
fig, ax = plt.subplots(figsize=(7, 3.2))
labels  = ['RIGHT', 'LEFT', 'UP', 'DOWN']
colors  = ['#3498DB', '#3498DB', '#3498DB', '#3498DB']
colors[2] = '#E74C3C'    # highlight the action that received three updates (a=2 = UP)
bars    = ax.bar(labels, Q[(0, 0, 0, 0, 0, 0)], color=colors, edgecolor='k')
for bar, val in zip(bars, Q[(0, 0, 0, 0, 0, 0)]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
            f'{val:.2f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Q(s, a)')
ax.set_title('Q-values for state (0,0,0,0,0,0) after three UP-action updates\n(other actions never selected -> Q = 0)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## 6.6 Experimental Configuration

Defaults reported below match the CLI defaults of `q_learn_improved.py`. Running the script unchanged reproduces the artifacts in `designs/q_improved/`:

| Parameter | Value | Source |
|---|---|---|
| Episodes ($N_{\text{ep}}$) | 200 | `N_EPISODES` |
| Max steps / episode | 34 | `MAX_STEPS` (matches GA chromosome length) |
| Learning rate $\alpha$ | 0.1 | `ALPHA` |
| Discount $\gamma$ | 0.9 | `GAMMA` |
| $\varepsilon_{\text{start}}$ | 1.0 | `EPSILON_START` |
| $\varepsilon_{\min}$ | 0.05 | `EPSILON_MIN` |
| $\varepsilon$ decay | $0.97$ multiplicative | `EPSILON_DECAY` |
| State bins per feature | 3 | `N_BINS` -> max state space $3^6 = 729$ |
| Random start each episode | True | `RANDOM_START` |
| Iron budget $[K_1, K_2]$ | $[80, 180]$ cells | `seqTO_env` |
| Reward scalar | mean $\lvert B \rvert \times 1000$ per step (cumulative over episode) | `seqTO_env.calculate_reward` |

$\varepsilon$ reaches its floor near episode $\log(0.05) / \log(0.97) \approx 98$, so roughly the first half of training is exploration-heavy and the second half is exploitation.

## 6.7 Run Log and Convergence

Running `python q_learn_improved.py` with the defaults above produces three artifacts in `designs/q_improved/`:

- `training_stats_improved.json` — per-episode reward, $\varepsilon$, and Q-table size for all 200 episodes
- `episodes/ep_NNN_mask.png` — rendered iron mask at the end of each episode (200 PNGs)
- `episodes/ep_NNN_final.fem` — solved FEMM model for each episode

The cell below loads `training_stats_improved.json` and renders:

1. Episode-reward curve (with running max) — shows convergence rate
2. $\varepsilon$ decay — confirms exploration schedule
3. Q-table size growth — confirms state-space coverage saturates well below 729 states
4. The iron mask of the highest-reward episode — the *output* of the run

In [ ]:
# Load and visualise the Q-learning run from designs/q_improved/
import json
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np

stats    = json.load(open('designs/q_improved/training_stats_improved.json'))
rewards  = stats['episode_rewards']
epsilons = stats['epsilon_history']
qsizes   = stats['q_size_history']
hp       = stats['hyperparams']
n_ep     = len(rewards)
best_ep  = int(np.argmax(rewards)) + 1            # 1-indexed for file naming
best_rew = max(rewards)

fig = plt.figure(figsize=(14, 8))
gs  = fig.add_gridspec(2, 3)

# Top-left: cumulative reward per episode
ax = fig.add_subplot(gs[0, :2])
ax.plot(range(1, n_ep + 1), rewards, 'steelblue', lw=0.7, alpha=0.7, label='Episode reward')
ax.plot(range(1, n_ep + 1), np.maximum.accumulate(rewards), 'crimson', lw=2.0, label='Best so far')
ax.axvline(best_ep, color='black', lw=0.8, ls='--', alpha=0.5)
ax.text(best_ep + 2, best_rew, f'  ep {best_ep}: {best_rew:.0f}', va='center', fontsize=8)
ax.set_xlabel('Episode'); ax.set_ylabel('Cumulative episode reward')
ax.set_title(f'Q-learning convergence -- {n_ep} episodes')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)

# Top-right: epsilon decay
ax = fig.add_subplot(gs[0, 2])
ax.plot(range(1, n_ep + 1), epsilons, 'darkorange', lw=1.5)
ax.axhline(hp['epsilon_min'], color='grey', ls=':', alpha=0.6,
           label=f"eps_min = {hp['epsilon_min']}")
ax.set_xlabel('Episode'); ax.set_ylabel('epsilon')
ax.set_title('Exploration rate schedule')
ax.legend(); ax.grid(alpha=0.3)

# Bottom-left: Q-table size growth
ax = fig.add_subplot(gs[1, 0])
ax.plot(range(1, n_ep + 1), qsizes, 'forestgreen', lw=1.5)
ax.axhline(hp['max_states'], color='grey', ls=':', alpha=0.6,
           label=f"max = {hp['max_states']} states")
ax.set_xlabel('Episode'); ax.set_ylabel('# states in Q-table')
ax.set_title('State-space coverage')
ax.legend(); ax.grid(alpha=0.3)

# Bottom-right (spanning 2 cols): best-episode iron mask
ax = fig.add_subplot(gs[1, 1:])
mask_img = mpimg.imread(f'designs/q_improved/episodes/ep_{best_ep:03d}_mask.png')
ax.imshow(mask_img)
ax.set_title(f'Best episode -- ep {best_ep}, reward {best_rew:.0f}')
ax.axis('off')

plt.suptitle('Q-Learning Results on C-core SeqTO-v1', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Total episodes        : {n_ep}')
print(f"Hyperparameters       : alpha={hp['alpha']}, gamma={hp['gamma']}, eps {hp['epsilon_start']}->{hp['epsilon_min']}")
print(f'Best episode          : {best_ep}   reward = {best_rew:.2f}')
print(f"Final Q-table size    : {qsizes[-1]}  /  {hp['max_states']} max states  (coverage: {100*qsizes[-1]/hp['max_states']:.1f} %)")
print(f'Final epsilon         : {epsilons[-1]:.4f}')

## 6.8 Best Topology vs the Genetic Algorithm

Chapter 5 reported the GA's best C-core topology (98 iron cells, fitness $1{,}628.61$ as mean $\lvert B \rvert \times 1000$ — see `designs/ga/ga_best_mask.npy`). The cell below puts the Q-learning best topology and the GA best topology side-by-side for visual comparison. The two scalars *aren't* directly comparable — the GA reports a single mean-$\lvert B \rvert$ snapshot while Q-learning reports a cumulative per-step reward — so the comparison here is structural (where does the agent place iron?) rather than numeric.

Force-in-Newtons benchmarking of both topologies via the Maxwell stress tensor is deferred to Section 7.

In [ ]:
# Side-by-side: Q-learning best topology vs GA best topology — both rendered
# with the same region-map palette so they are visually comparable.
#
# The GA stores its best mask as a numpy .npy (Chapter 5); the Q-learning run
# only saved per-episode .fem files (FEMM format) and PNG renderings.  To
# render the Q-learning mask in the same style as the GA panel we recover
# the iron mask by parsing the FEMM file's block-label table.

import json
import re
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.colors import ListedColormap

# Geometry
ROWS, COLS = 18, 35
COIL_LEN, COIL1_COLS, COIL2_COLS = 9, (2, 5), (11, 14)
ARM_ROWS, ARM_COLS = (0, 15), (27, 33)
DW_ROWS, DW_COLS   = (0, 15), (5, 26)


def parse_fem_iron_mask(fem_path):
    """Read a FEMM .fem file and return the iron mask (18x35).

    Block labels in the [NumBlockLabels] section have format
        x  y  material_idx  ...
    where material_idx refers into [BlockProps] (1=Air, 2=Iron, 3=Copper for
    this project).  FEMM's first coord ('x' in millimetres) corresponds to
    *row* in seqTO_env indexing; the second ('y') corresponds to *column*.
    Cell centres sit at half-integers, so int(coord) gives the cell index.
    Returns the FULL iron mask including the armature pillar.
    """
    with open(fem_path) as f:
        text = f.read()
    n_match = re.search(r'\[NumBlockLabels\]\s*=\s*(\d+)', text)
    n = int(n_match.group(1))
    block_lines = text[n_match.end():].splitlines()[1:n + 1]

    mask = np.zeros((ROWS, COLS), dtype=int)
    for line in block_lines:
        parts = line.split()
        if len(parts) < 3:
            continue
        i, j, mat = int(float(parts[0])), int(float(parts[1])), int(parts[2])
        if mat == 2 and 0 <= i < ROWS and 0 <= j < COLS:
            mask[i, j] = 1
    return mask


def _is_design(i, j):
    return (DW_ROWS[0] <= i < DW_ROWS[1] and DW_COLS[0] <= j < DW_COLS[1]
            and not (i < COIL_LEN and COIL2_COLS[0] <= j < COIL2_COLS[1]))


def build_region_map(iron_mask):
    """Map (iron + structural geometry) -> integer regions for plotting."""
    region = np.zeros((ROWS, COLS), dtype=int)
    for i in range(ROWS):
        for j in range(COLS):
            if i < COIL_LEN and COIL1_COLS[0] <= j < COIL1_COLS[1]:    region[i, j] = 1
            elif i < COIL_LEN and COIL2_COLS[0] <= j < COIL2_COLS[1]:  region[i, j] = 2
            elif ARM_ROWS[0] <= i < ARM_ROWS[1] and ARM_COLS[0] <= j < ARM_COLS[1]: region[i, j] = 3
            elif _is_design(i, j):                                      region[i, j] = 4 if iron_mask[i, j] else 5
            elif DW_ROWS[0] <= i < DW_ROWS[1] and DW_COLS[0] <= j < DW_COLS[1]: region[i, j] = 6
    return region


# ── Load both best masks ──
stats   = json.load(open('designs/q_improved/training_stats_improved.json'))
best_ep = int(np.argmax(stats['episode_rewards'])) + 1
ql_full_mask = parse_fem_iron_mask(f'designs/q_improved/episodes/ep_{best_ep:03d}_final.fem')

# Restrict to design window — armature is structural, not GA/QL-placed
ql_deposited = ql_full_mask.copy()
ql_deposited[ARM_ROWS[0]:ARM_ROWS[1], ARM_COLS[0]:ARM_COLS[1]] = 0   # zero out armature
ql_iron_count = int(ql_deposited.sum())
ql_reward     = max(stats['episode_rewards'])

ga_mask    = np.load('designs/ga/ga_best_mask.npy')
ga_logbook = json.load(open('designs/ga/ga_logbook.json'))
ga_fit     = max(r['max'] for r in ga_logbook)
ga_iron_count = int(ga_mask.sum())

# ── Render both with the same palette ──
CLRS = ['#D5D8DC', '#2980B9', '#E67E22', '#27AE60', '#2C3E50', '#FEF9E7', '#FAD7A0']

fig, axes = plt.subplots(2, 1, figsize=(11, 7.5))

axes[0].imshow(build_region_map(ql_deposited),
               cmap=ListedColormap(CLRS), vmin=0, vmax=6, origin='upper', aspect='equal')
axes[0].set_title(f'Q-learning best — ep {best_ep}, {ql_iron_count} deposited iron cells, '
                  f'cumulative reward = {ql_reward:.0f}',
                  fontsize=11)
axes[0].set_xlabel('column'); axes[0].set_ylabel('row')

axes[1].imshow(build_region_map(ga_mask),
               cmap=ListedColormap(CLRS), vmin=0, vmax=6, origin='upper', aspect='equal')
axes[1].set_title(f'GA best — {ga_iron_count} iron cells, fitness = {ga_fit:.2f} (mean |B|*1000)',
                  fontsize=11)
axes[1].set_xlabel('column'); axes[1].set_ylabel('row')
axes[1].legend(handles=[
    mpatches.Patch(fc='#2C3E50', label='Iron (deposited)'),
    mpatches.Patch(fc='#FEF9E7', ec='grey', lw=0.5, label='Air (design window)'),
    mpatches.Patch(fc='#2980B9', label='Left coil'),
    mpatches.Patch(fc='#E67E22', label='Right coil'),
    mpatches.Patch(fc='#27AE60', label='Armature'),
], fontsize=8, loc='upper right')

plt.suptitle('SeqTO-v1 -- Q-learning vs Genetic Algorithm (best topologies)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('Iron-cell counts (deposited in design window only):')
print(f'  Q-learning best : {ql_iron_count} cells  (cumulative episode reward {ql_reward:.0f})')
print(f'  GA best         : {ga_iron_count} cells  (fitness {ga_fit:.2f}, single mean |B| snapshot)')
print('\nReward scales differ -- compare topology *shape*, not magnitude.')
print('The Q-learning result here is the cumulative-reward winner, not necessarily')
print('the best single-step armature-flux topology; Newton-force benchmarking via')
print('the Maxwell stress tensor is deferred to Section 7.')

## 6.10 Application to the SynRM Environment

The same compact 6-feature Q-learning approach transfers to the polar SynRM rotor with the script [`q_learn_synrm.py`](q_learn_synrm.py). Two adaptations from the C-core script:

1. **Polar action semantics.** RIGHT/LEFT increment / decrement the angular index $j$, UP/DOWN increment / decrement the radial index $i$ (UP = radially outward toward the air gap). Same as the GA in Chp 5 Section 5.7.
2. **Customisable grid size.** `q_learn_synrm.py --grid-size N` for any $N \in [5, 10]$, using the `seqTO_SynRM_env.configure()` rescaling introduced in Section 5.7.5. The env auto-builds `synrm_seqto_NxN.fem` on first run and rescales the iron budget ($K_1 = \max(N, 5)$, $K_2 = \lfloor 0.8 \, N^2 \rfloor$) and `MAX_STEPS = N^2`.

The 6-feature state is adapted for polar geometry (Section 5.7's analogue):

| # | Feature | What it captures |
|---|---|---|
| 1 | `pos_r_bin`    | Controller radial index (3 bins) |
| 2 | `pos_c_bin`    | Controller angular index (3 bins) |
| 3 | `iron_frac`    | Iron budget consumed relative to $K_2$ |
| 4 | `outer_iron`   | Fraction of iron in the outermost ~40 % of radial rows |
| 5 | `left_iron`    | Fraction of iron in the leftmost ~40 % of angular cols (q-axis side) |
| 6 | `step_bin`     | Episode progress |

Features 4 and 5 capture *where in polar space* the iron sits — the diagnostic in Chapter 6 (Section 6.7) and the GA's best topology in 5.7.3 both pointed at the outer-radial and q-axis-side cells as the strongest reward levers.

### 6.10.1 Initial 35-episode run at 7×7

An initial training run completed 35 of 75 planned episodes before being interrupted:

```bash
python q_learn_synrm.py --grid-size 7 --episodes 75    # 35 episodes captured
```

The artefacts in `designs/q_synrm_7x7_improved/` are the truncated run. The cell below loads them and renders convergence + best-episode topology inline. A full 200-episode rerun is identified as planned future work in Section 7.6.1.

In [ ]:
# Load and visualise the SynRM Q-learning run from designs/q_synrm_7x7_improved/
import json
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np

# Polar geometry (matches seqTO_SynRM_env.configure(7, 7))
DD_ROWS_S, DD_COLS_S = 7, 7
R_HUB, R_PERI_IN     = 12.0, 36.7
DR_S       = (R_PERI_IN - R_HUB) / DD_ROWS_S    # 3.529 mm
DTHETA_S   = 90.0 / DD_COLS_S                   # 12.86 deg

stats = json.load(open('designs/q_synrm_7x7_improved/training_stats_improved.json'))
rew   = stats['episode_rewards']
eps   = stats['epsilon_history']
qsz   = stats['q_size_history']
hp    = stats['hyperparams']
n_ep  = len(rew)
best_ep  = int(np.argmax(rew)) + 1
best_rew = max(rew)

fig = plt.figure(figsize=(14, 5))
ax_conv = fig.add_subplot(1, 2, 1)
ax_topo = fig.add_subplot(1, 2, 2, projection='polar')

# Left: episode reward + running max + epsilon (twin axis)
ax_conv.plot(range(1, n_ep + 1), rew, 'steelblue', lw=1.0, marker='o',
             markersize=5, label='Episode reward')
ax_conv.plot(range(1, n_ep + 1), np.maximum.accumulate(rew),
             'crimson', lw=2.0, label='Best so far')
ax_conv.axvline(best_ep, color='black', lw=0.6, ls='--', alpha=0.5)
ax_conv.text(best_ep + 0.1, best_rew, f' ep {best_ep}: {best_rew:.0f}',
             va='center', fontsize=8)
ax_conv.set_xlabel('Episode'); ax_conv.set_ylabel('Cumulative reward (max |torque|)')
ax_conv.set_title(f'SynRM Q-learning ({DD_ROWS_S}x{DD_COLS_S} grid) -- {n_ep} episodes')
ax_conv.legend(loc='upper right'); ax_conv.grid(alpha=0.3)

ax2 = ax_conv.twinx()
ax2.plot(range(1, n_ep + 1), eps, 'darkorange', lw=1.0, alpha=0.6, label='epsilon')
ax2.set_ylabel('epsilon (right axis)', color='darkorange'); ax2.set_ylim(0, 1)
ax2.tick_params(axis='y', labelcolor='darkorange')

# Right: best-episode polar topology (loaded as the pre-rendered PNG from training)
mask_img = mpimg.imread(f'designs/q_synrm_7x7_improved/episodes/ep_{best_ep:03d}_mask.png')
ax_topo.remove()
ax_topo = fig.add_subplot(1, 2, 2)
ax_topo.imshow(mask_img); ax_topo.axis('off')
ax_topo.set_title(f'Best episode mask (ep {best_ep}, reward {best_rew:.1f})', fontsize=10)

plt.suptitle('SynRM Q-Learning (7x7) -- 35-episode initial run (of 75 planned)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"Grid              : {hp['dd_rows']}x{hp['dd_cols']}")
print(f"Episodes          : {n_ep}    (max_steps/ep = {hp['max_steps']})")
print(f"Hyperparameters   : alpha={hp['alpha']}  gamma={hp['gamma']}  "
      f"eps {hp['epsilon_start']}->{hp['epsilon_min']}  decay {hp['epsilon_decay']}")
print(f"Best episode      : {best_ep}  reward={best_rew:.2f}  eps={eps[best_ep-1]:.3f}")
print(f"Final Q-table     : {qsz[-1]}  /  {hp['max_states']} max states  "
      f"({100*qsz[-1]/hp['max_states']:.1f}% coverage)")
print(f"FEMM calls        : {n_ep * hp['max_steps']}")

### 6.10.2 What this initial run does and doesn't tell us

**What it confirms:**
- `q_learn_synrm.py --grid-size 7` runs end-to-end. The env reconfigures correctly ($DR = 3.529$ mm, $D\theta = 12.86°$, $K_1 = 7$, $K_2 = 39$, `MAX_STEPS` = 49), auto-builds `synrm_seqto_7x7.fem` on first run, FEMM solves, the Q-dict grows, training artefacts are saved.
- The grid-customisation refactor from Section 5.7.5 propagates through to Q-learning unchanged.
- The $\varepsilon$-decay schedule does drive $\varepsilon$ from 0.92 down to $\sim 0.05$ over the 35 captured episodes, so by the tail of the run the agent is mostly exploiting rather than exploring.

**What it doesn't yet show:**
- Convergence. The "best episode" at $t = 3$ was found with $\varepsilon = 0.78$ — meaning 78 % of actions were still random. That is a lucky random walk, not a learned-policy result. Subsequent episodes (with lower $\varepsilon$) produced rewards in the 135–210 range, *below* the gen-3 high of 249.
- State-space coverage. Only $111 / 729 = 15 \%$ of the compact-state space was visited across 35 episodes. With this little coverage the Q-table cannot have meaningfully converged.
- Comparability with the SynRM GA (fitness 11.29 at 5×5, Section 5.7.3). The two are on different scales (cumulative-over-49-steps vs single-snapshot-max-torque) *and* different grids.

**Planned future work.** A full 200-episode rerun (`python q_learn_synrm.py --grid-size 7 --episodes 200`) is identified as the priority follow-up — wall-clock is the only constraint, the script and env are already in place. When that completes, re-executing this notebook in place (`jupyter nbconvert --to notebook --execute --inplace 06_mdp_implementation.ipynb`) will pick up the new `training_stats_improved.json` and replace both the figure above and the caveat paragraph in this section with a convergence assessment. The placeholder is intentional — Chapter 4 is presented as complete *with the data currently available*, and the rerun is tracked as Item 1 in the deferred-work list (Section 7.6.1).

## 6.11 Summary

Tabular Q-learning applied to SeqTO-v1:

1. **Compact state representation enables convergence.** A hand-crafted 6-feature state ($3^6 = 729$ max states) gives the agent enough Markov structure to learn while keeping the Q-table small enough that each state is visited many times across 200 episodes.
2. **Same FEMM reward as the GA.** `seqTO_env.calculate_reward()` (and `seqTO_SynRM_env.calculate_reward()` for the polar variant) is the only reward signal — no surrogate model, no mock evaluator. The C-core training run was completed via `python q_learn_improved.py`.
3. **$\varepsilon$-greedy with multiplicative decay** balances exploration (first ~100 episodes) and exploitation (remaining ~100). The C-core Q-table saturates well below the 729-state ceiling, confirming the state-space is well-sized for the problem.
4. **Convergent C-core training** — the running-max reward curve climbs over the first ~80 episodes and then plateaus, mirroring the GA's stall behaviour in Chapter 5 but reached through experience accumulation rather than population genetics.
5. **Same problem, two solvers.** The C-core best topologies found by GA (Chapter 5) and Q-learning are placed side-by-side in Section 6.8.
6. **SynRM extension** (Section 6.10). The same compact-state Q-learning transfers to the polar SynRM rotor via `q_learn_synrm.py`, which inherits the `--grid-size N` flag introduced in Section 5.7.5. A 35-episode initial run at $7 \times 7$ (of 75 planned) confirms the plumbing works end-to-end but is short of convergence — only 15 % of the compact state-space was visited. A full 200-episode rerun is flagged as planned future work (Section 7.6.1).

Numeric reward scales differ between GA and Q-learning (snapshot vs cumulative), and Q-learning on SynRM is not yet converged, so the cross-method comparison is structural rather than directly numeric. Newton-force benchmarking and a direct numeric comparison are deferred to Section 7.